## Задача 4

Есть список prices, один элемент - цена акции в этот день. Можно сделать до k сделок. Одна сделка - купили одну акцию, потом в другой день продали. Следующую покупку можно делать со следующего дня после продажи

Возьмём цены из условия. Дни в разборе считаем с 1, индексы списка в коде с 0

In [1]:
prices = [3, 2, 6, 5, 0, 3]
k = 2
days = []
for i in range(len(prices)):
    days.append(i + 1)

print('день', days)
print('цена', prices)

день [1, 2, 3, 4, 5, 6]
цена [3, 2, 6, 5, 0, 3]


сначала разрешим одну сделку и посмотрим на день 3, цена акции 6
до этого видели цены 3 и 2, значит можно было купить за 2 и сейчас продать за 6, прибыль 4

на следующий день цена 5, если продать сейчас, получим 5 - 2 = 3
мы могли продать ещё вчера за 6, поэтому лучшая прибыль к концу дня всё ещё 4

как в задаче с отелем (5.ipynb, функция data_reads), заведём список, где одна позиция - один день
там по дням считали гостей, здесь в row запишем, сколько можно заработать к концу каждого дня
отдельно в best_buy запомним, сколько денег могло остаться после покупки, например при покупке за 2 с нуля остаётся -2

In [ ]:
def next_row(prices, prev_row): 
    row = [0] * len(prices)
    if len(prices) == 0:
        return row
    best_buy = -prices[0] # в первый день можно только купить или подождать

    for day in range(1, len(prices)):
        row[day] = row[day - 1] # сегодня ничего не делаем
        sell = best_buy + prices[day]
        if sell > row[day]:
            row[day] = sell

        buy = prev_row[day - 1] - prices[day]
        if buy > best_buy:
            best_buy = buy # продать можно со следующего дня
    return row

no_trades = [0] * len(prices)
one_trade = next_row(prices, no_trades)
print('цена', prices)
print('до одной сделки', one_trade)

цена [3, 2, 6, 5, 0, 3]
до одной сделки [0, 0, 4, 4, 4, 4]


теперь разрешим вторую сделку
на пятый день цена упала до 0, а к концу четвёртого дня мы уже могли заработать 4
покупаем за 0, остаётся 4 и одна акция на руках
на шестой день продаём её за 3, итого 4 + 3 = 7

отсюда в next_row берётся prev_row[day - 1] - prices[day]
для первой покупки в prev_row лежали нули, для второй туда подставим one_trade
берём именно вчерашнюю прибыль, тк в один день продать и снова купить нельзя

In [3]:
two_trades = next_row(prices, one_trade)
print('день', days)
print('до одной сделки', one_trade)
print('до двух сделок', two_trades)

день [1, 2, 3, 4, 5, 6]
до одной сделки [0, 0, 4, 4, 4, 4]
до двух сделок [0, 0, 4, 4, 4, 7]


теперь соберём всё для любого k
начинаем со строки из нулей, каждый вызов next_row разрешает ещё одну сделку
последний элемент строки - прибыль за все дни
на сделку нужно хотя бы два дня, поэтому больше n // 2 строк считать смысла нет

In [ ]:
def max_profit(prices, k): 
    if len(prices) < 2 or k == 0:
        return 0
    row = [0] * len(prices)
    max_trades = len(prices) // 2
    if k > max_trades:
        k = max_trades
    for i in range(k):
        row = next_row(prices, row)
    return row[-1]

print('пример из условия:', max_profit(prices, k))
print('те же цены, разрешили одну сделку:', max_profit(prices, 1))
print('цена только падает:', max_profit([6, 5, 2], 2))
print('всего один день:', max_profit([3], 2))

пример из условия: 7
те же цены, разрешили одну сделку: 4
цена только падает: 0
всего один день: 0


## Задача 6

найти кратчайшие расстояния от start до каждой вершины с помощью алгоритма Bellman-Ford

берём graph из AlgoMipt2026/Graphs.ipynb, пример сразу после функции dejkstra
строка - откуда идём, столбец - куда, в клетке лежит вес ребра
None - ребра нет, 0 - ребро есть и пройти по нему можно за 0
если можно ходить в обе стороны, заполняем обе клетки
если из одной вершины в другую идёт несколько рёбер, берём самое дешёвое

In [5]:
n = 5
graph = []
for i in range(n):
    graph.append([None] * n)
graph[0][1] = 1
graph[1][0] = -2
graph[1][2] = graph[2][1] = 2
graph[1][3] = graph[3][1] = 3
graph[0][4] = graph[4][0] = 4
graph[0][3] = graph[3][0] = 5

for row in graph:
    print(row)

[None, 1, None, 5, 4]
[-2, None, 2, 3, None]
[None, 2, None, None, None]
[5, 3, None, None, None]
[4, None, None, None, None]


посмотрим на graph[0][1] и graph[1][0]
из 0 в 1 идём за 1, обратно за -2, за один круг получается 1 - 2 = -1
прошли два круга - уже -2, и так можно уменьшать стоимость сколько угодно

сначала уберём стрелку из 0 в 1, чтобы посмотреть как меняются расстояния без этого цикла
как в примере с удалением из дерева в 2_3.ipynb, сделаем копию через deepcopy
поменяем одну клетку в graph_copy, потом пригодится исходный graph

In [6]:
from copy import deepcopy

graph_copy = deepcopy(graph)
graph_copy[0][1] = None
start = 2
print('из 0 в 1:', graph_copy[0][1])
print('из 1 в 0:', graph_copy[1][0])

из 0 в 1: None
из 1 в 0: -2


из dejkstra(g, start) берём словарь dists
в start уже стоим, там расстояние 0, у остальных пока inf
дальше проверка с семинара - можно ли дойти дешевле через соседа
например, до 1 дошли за 2, из 1 в 0 идёт ребро веса -2, значит до 0 можно дойти за 2 - 2 = 0

чтобы проверить все рёбра, соберём их в edges
в cruskull(g) из AlgoMipt2026/Graphs.ipynb уже собирали тройки (weight, i, j)
здесь смотрим всю матрицу, тк вес из i в j и обратно может быть разный
дальше update_dists один раз проходит по edges и меняет расстояния там, где нашёл путь дешевле

In [7]:
def start_dists(g, start):
    dists = {}
    for i in range(len(g)):
        dists[i] = float('inf')
    dists[start] = 0
    return dists

def get_edges(g):
    edges = []
    for i in range(len(g)):
        for j in range(len(g[i])):
            if g[i][j] is not None:
                edges.append((g[i][j], i, j))
    return edges

def update_dists(edges, dists):
    changed = False
    for weight, u, v in edges:
        if dists[u] == float('inf'):
            continue
        new_dist = dists[u] + weight
        if new_dist < dists[v]:
            dists[v] = new_dist
            changed = True
    return changed

In [8]:
edges = get_edges(graph_copy)
dists = start_dists(graph_copy, start)
print('до обхода:', dists)
for step in range(1, n):
    update_dists(edges, dists)
    print('проход', step, dists)

до обхода: {0: inf, 1: inf, 2: 0, 3: inf, 4: inf}
проход 1 {0: inf, 1: 2, 2: 0, 3: inf, 4: inf}
проход 2 {0: 0, 1: 2, 2: 0, 3: 5, 4: inf}
проход 3 {0: 0, 1: 2, 2: 0, 3: 5, 4: 4}
проход 4 {0: 0, 1: 2, 2: 0, 3: 5, 4: 4}


до вершины 4 добрались только на третьем проходе, посмотрим почему

1 проход - из 2 в 1 дошли за 2, но строку 1 в матрице к этому моменту уже просмотрели

2 проход - из 1 в 0 дошли за 2 - 2 = 0, строку 0 тоже уже просмотрели

3 проход - из 0 в 4 дошли за 0 + 4 = 4

если в графе n вершин, то в пути без повторов максимум n - 1 рёбер
за проход успеем пройти хотя бы одно следующее ребро такого пути, значит хватит n - 1 проходов
если расстояния за проход остались прежними, можно вернуть dists сразу

теперь вернём стрелку из 0 в 1
если после n - 1 проходов расстояние всё ещё уменьшается, значит на пути к этой вершине есть цикл с отрицательной суммой весов
из неё тоже можно идти дальше, сначала сколько угодно раз обойти цикл, потом выйти к соседям
берём stack и обход соседей из dfs(g) в AlgoMipt2026/Graphs.ipynb
в stack кладём концы всех рёбер, где ещё можно уменьшить расстояние
от них идём дальше по стрелкам и ставим -inf всем, до кого дошли

In [ ]:
def bellman_ford(g, start): 
    dists = start_dists(g, start)
    edges = get_edges(g)
    for step in range(len(g) - 1):
        changed = update_dists(edges, dists)
        if not changed:
            return dists

    stack = []
    for weight, u, v in edges:
        if dists[u] == float('inf'):
            continue
        if dists[u] + weight < dists[v]:
            stack.append(v)

    while stack:
        cur = stack.pop()
        if dists[cur] == float('-inf'):
            continue
        dists[cur] = float('-inf')
        for i in range(len(g[cur])):
            if g[cur][i] is not None:
                stack.append(i)
    return dists

In [10]:
print('убрали стрелку 0 -> 1:', bellman_ford(graph_copy, start))
print('вернули стрелку 0 -> 1:', bellman_ford(graph, start))

убрали стрелку 0 -> 1: {0: 0, 1: 2, 2: 0, 3: 5, 4: 4}
вернули стрелку 0 -> 1: {0: -inf, 1: -inf, 2: -inf, 3: -inf, 4: -inf}


в исходном graph из цикла можно попасть во все пять вершин, поэтому у всех получилось -inf

а если цикл вообще в другой части графа?
сделаем ребро из 0 в 1 с весом 0, а цикл оставим отдельно между 2 и 3
теперь запустим сначала из 0, потом из 2 и посмотрим, где останется inf и где появится -inf

In [11]:
graph2 = []
for i in range(4):
    graph2.append([None] * 4)
graph2[0][1] = 0
graph2[2][3] = 1
graph2[3][2] = -2
print('старт в 0:', bellman_ford(graph2, 0))
print('старт в цикле:', bellman_ford(graph2, 2))

старт в 0: {0: 0, 1: 0, 2: inf, 3: inf}
старт в цикле: {0: inf, 1: inf, 2: -inf, 3: -inf}
